# Dataset Preprocessing for Restaurant ABSA

This notebook converts the raw SemEval-2014 Task 4 restaurant XML dataset into clean CSV files for this project.

Project requirements:

- Domain: restaurant reviews
- Fixed aspects: `Food`, `Service`, `Price`, `Eating Environment / Ambiance`
- Fixed labels: `Positive`, `Negative`, `Unknown`
- Every review must have all four aspect labels, even when an aspect is not mentioned.

## Step 1: Import Libraries and Define File Paths

The raw XML file stays in `backends/data/raw/`. The processed CSV files will be written to `backends/data/processed/`.

In [7]:
from pathlib import Path
import csv
import re
import xml.etree.ElementTree as ET
from collections import Counter

try:
    import pandas as pd
except ImportError as exc:
    raise RuntimeError("This notebook needs pandas. Install it with: pip install pandas") from exc

DATA_DIR = Path.cwd()
RAW_XML_PATH = DATA_DIR / "raw" / "Restaurants_Train_v2.xml"
PROCESSED_DIR = DATA_DIR / "processed"
WIDE_OUTPUT_PATH = PROCESSED_DIR / "restaurant_absa_4_aspects.csv"
LONG_OUTPUT_PATH = PROCESSED_DIR / "restaurant_absa_aspect_rows.csv"

print("Notebook working directory:", DATA_DIR)
print("Raw XML path:", RAW_XML_PATH)
print("Wide output path:", WIDE_OUTPUT_PATH)
print("Long output path:", LONG_OUTPUT_PATH)

assert RAW_XML_PATH.exists(), f"Missing raw dataset: {RAW_XML_PATH}"

Notebook working directory: /Users/kafe/Desktop/NLP_Final Project/absa_v01/backends/data
Raw XML path: /Users/kafe/Desktop/NLP_Final Project/absa_v01/backends/data/raw/Restaurants_Train_v2.xml
Wide output path: /Users/kafe/Desktop/NLP_Final Project/absa_v01/backends/data/processed/restaurant_absa_4_aspects.csv
Long output path: /Users/kafe/Desktop/NLP_Final Project/absa_v01/backends/data/processed/restaurant_absa_aspect_rows.csv


## Step 2: Define Project Label Mapping

SemEval uses lowercase category and polarity names. This project uses four fixed aspect names and three output labels.

`anecdotes/miscellaneous` is ignored because it does not belong to the four project aspects.

In [8]:
PROJECT_ASPECTS = [
    "Food",
    "Service",
    "Price",
    "Eating Environment / Ambiance",
]

CATEGORY_MAP = {
    "food": "Food",
    "service": "Service",
    "price": "Price",
    "ambience": "Eating Environment / Ambiance",
}

POLARITY_MAP = {
    "positive": "Positive",
    "negative": "Negative",
    "neutral": "Unknown",
    "conflict": "Unknown",
}

def clean_text(text: str) -> str:
    """Normalize whitespace while preserving the original review wording."""
    return re.sub(r"\s+", " ", text or "").strip()

def merge_sentiment(existing: str, new_value: str) -> str:
    """Merge repeated labels for the same project aspect in one sentence.

    If two labels disagree, the project cannot represent mixed sentiment directly,
    so we use Unknown instead of inventing a fourth label.
    """
    if existing == "Unknown":
        return new_value
    if new_value == "Unknown" or existing == new_value:
        return existing
    return "Unknown"

print("Project aspects:", PROJECT_ASPECTS)
print("Category mapping:", CATEGORY_MAP)
print("Polarity mapping:", POLARITY_MAP)

Project aspects: ['Food', 'Service', 'Price', 'Eating Environment / Ambiance']
Category mapping: {'food': 'Food', 'service': 'Service', 'price': 'Price', 'ambience': 'Eating Environment / Ambiance'}
Polarity mapping: {'positive': 'Positive', 'negative': 'Negative', 'neutral': 'Unknown', 'conflict': 'Unknown'}


## Step 3: Parse the XML and Build a Wide Dataset

The wide dataset has one row per review. Each row contains all four project aspects. Missing aspects are filled with `Unknown`.

In [9]:
tree = ET.parse(RAW_XML_PATH)
root = tree.getroot()

wide_rows = []
ignored_categories = Counter()
raw_category_counts = Counter()
raw_polarity_counts = Counter()

for sentence in root.findall("sentence"):
    sentence_id = sentence.attrib.get("id", "")
    review = clean_text(sentence.findtext("text"))
    labels = {aspect: "Unknown" for aspect in PROJECT_ASPECTS}

    for category_node in sentence.findall("./aspectCategories/aspectCategory"):
        raw_category = category_node.attrib.get("category", "").strip().lower()
        raw_polarity = category_node.attrib.get("polarity", "").strip().lower()
        raw_category_counts[raw_category] += 1
        raw_polarity_counts[raw_polarity] += 1

        project_aspect = CATEGORY_MAP.get(raw_category)
        if project_aspect is None:
            ignored_categories[raw_category] += 1
            continue

        project_sentiment = POLARITY_MAP.get(raw_polarity, "Unknown")
        labels[project_aspect] = merge_sentiment(labels[project_aspect], project_sentiment)

    wide_rows.append({
        "id": sentence_id,
        "review": review,
        **labels,
    })

wide_df = pd.DataFrame(wide_rows, columns=["id", "review", *PROJECT_ASPECTS])

print("Raw sentences:", len(wide_df))
print("Raw SemEval category counts:", dict(raw_category_counts))
print("Raw SemEval polarity counts:", dict(raw_polarity_counts))
print("Ignored categories:", dict(ignored_categories))
wide_df.head()

Raw sentences: 3041
Raw SemEval category counts: {'service': 597, 'food': 1232, 'anecdotes/miscellaneous': 1132, 'price': 321, 'ambience': 431}
Raw SemEval polarity counts: {'negative': 839, 'positive': 2179, 'conflict': 195, 'neutral': 500}
Ignored categories: {'anecdotes/miscellaneous': 1132}


,id,review,Food,Service,Price,Eating Environment / Ambiance
0,3121,But the staff was so horrible to us.,Unknown,Negative,Unknown,Unknown
1,2777,"To be completely fair, the only redeeming fact...",Positive,Unknown,Unknown,Unknown
2,1634,"The food is uniformly exceptional, with a very...",Positive,Unknown,Unknown,Unknown
3,2534,Where Gabriela personaly greets you and recomm...,Unknown,Positive,Unknown,Unknown
4,583,"For those that go once and don't enjoy it, all...",Unknown,Unknown,Unknown,Unknown


## Step 4: Create a Long Aspect-Pair Dataset

The long dataset has four rows per review: one row for each `(review, aspect)` pair. This is useful for traditional ML and BERT-style aspect-pair classification.

In [10]:
long_rows = []

for _, row in wide_df.iterrows():
    for aspect in PROJECT_ASPECTS:
        long_rows.append({
            "id": row["id"],
            "review": row["review"],
            "aspect": aspect,
            "sentiment": row[aspect],
        })

long_df = pd.DataFrame(long_rows, columns=["id", "review", "aspect", "sentiment"])

print("Long rows:", len(long_df))
print("Expected rows:", len(wide_df) * len(PROJECT_ASPECTS))
long_df.head(8)

Long rows: 12164
Expected rows: 12164


,id,review,aspect,sentiment
0,3121,But the staff was so horrible to us.,Food,Unknown
1,3121,But the staff was so horrible to us.,Service,Negative
2,3121,But the staff was so horrible to us.,Price,Unknown
3,3121,But the staff was so horrible to us.,Eating Environment / Ambiance,Unknown
4,2777,"To be completely fair, the only redeeming fact...",Food,Positive
5,2777,"To be completely fair, the only redeeming fact...",Service,Unknown
6,2777,"To be completely fair, the only redeeming fact...",Price,Unknown
7,2777,"To be completely fair, the only redeeming fact...",Eating Environment / Ambiance,Unknown


## Step 5: Validate the Processed Data

These checks make sure the processed files match the project rules before saving.

In [11]:
ALLOWED_LABELS = {"Positive", "Negative", "Unknown"}

assert not wide_df.empty, "The wide dataset is empty."
assert not long_df.empty, "The long dataset is empty."
assert wide_df["id"].notna().all(), "Some rows are missing IDs."
assert wide_df["review"].str.len().gt(0).all(), "Some rows have empty review text."
assert list(wide_df.columns) == ["id", "review", *PROJECT_ASPECTS], "Wide dataset columns are incorrect."
assert list(long_df.columns) == ["id", "review", "aspect", "sentiment"], "Long dataset columns are incorrect."
assert set(long_df["aspect"].unique()) == set(PROJECT_ASPECTS), "Long dataset has unexpected aspects."
assert set(long_df["sentiment"].unique()).issubset(ALLOWED_LABELS), "Long dataset has unexpected labels."

print("Wide shape:", wide_df.shape)
print("Long shape:", long_df.shape)
print("Wide label distribution:")
print(wide_df[PROJECT_ASPECTS].stack().value_counts())
print("\nLong label distribution:")
print(long_df["sentiment"].value_counts())

Wide shape: (3041, 6)
Long shape: (12164, 4)
Wide label distribution:
Unknown     9891
Positive    1633
Negative     640
Name: count, dtype: int64

Long label distribution:
sentiment
Unknown     9891
Positive    1633
Negative     640
Name: count, dtype: int64


## Step 6: Save the Processed CSV Files

The wide file is convenient for the Flutter/API output format. The long file is convenient for model training and evaluation.

In [12]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

wide_df.to_csv(WIDE_OUTPUT_PATH, index=False, quoting=csv.QUOTE_MINIMAL)
long_df.to_csv(LONG_OUTPUT_PATH, index=False, quoting=csv.QUOTE_MINIMAL)

print("Saved wide dataset:", WIDE_OUTPUT_PATH)
print("Saved long dataset:", LONG_OUTPUT_PATH)
print("Wide file size:", WIDE_OUTPUT_PATH.stat().st_size, "bytes")
print("Long file size:", LONG_OUTPUT_PATH.stat().st_size, "bytes")

Saved wide dataset: /Users/kafe/Desktop/NLP_Final Project/absa_v01/backends/data/processed/restaurant_absa_4_aspects.csv
Saved long dataset: /Users/kafe/Desktop/NLP_Final Project/absa_v01/backends/data/processed/restaurant_absa_aspect_rows.csv
Wide file size: 340709 bytes
Long file size: 1212881 bytes
